<a href="https://colab.research.google.com/github/gmauricio-toledo/tda-gdl/blob/main/01-Maldici%C3%B3n_de_la_dimensionalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Maldición de la dimensionalidad

La maldición de la dimensionalidad es un fenómeno fundamental que emerge cuando trabajamos con espacios de alta dimensión, donde las intuiciones geométricas desarrolladas en dimensiones bajas fallan. Conforme aumenta la dimensión del espacio, las distancias entre puntos tienden a volverse uniformemente grandes, los volúmenes se concentran en las esquinas de los hipercubos, y los algoritmos que dependen de nociones de proximidad pierden su eficacia. Este comportamiento contraintuitivo no es meramente una curiosidad matemática, sino que plantea desafíos profundos en áreas como el análisis de datos, la optimización, y el aprendizaje automático, donde frecuentemente debemos trabajar con espacios de dimensión exponencialmente grande. La comprensión rigurosa de estos fenómenos requiere herramientas de la teoría de la medida, probabilidad en espacios de alta dimensión, y geometría convexa, revelando conexiones inesperadas entre la combinatoria, el análisis funcional y la geometría diferencial.

---

En este experimento generaremos puntos aleatorios en espacios de diferentes dimensiones y calcularemos las distancias euclidianas entre ellos. Conforme aumenta la dimensionalidad, observaremos cómo todas las distancias tienden a volverse muy similares entre sí, perdiendo la capacidad de discriminar entre puntos "cercanos" y "lejanos". Este fenómeno, conocido como la maldición de la dimensionalidad, explica por qué muchos algoritmos de machine learning (como k-NN o clustering) pierden efectividad en espacios de alta dimensión.

In [ ]:
#@title Experimento maldición de la dimensionalidad
import numpy as np
# from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

side = 2  # Longitud del lado del hipercubo
n_puntos = 100
dims = list(range(2, 500, 3))
n_repetitions = 5  # Repeticiones por cada experimento

average_distances = []
min_distances = []
max_distances = []
std_distances = []
relative_std = []  # Desviación estándar relativa

for i, dim in enumerate(dims):
    exp_avg, exp_min, exp_max, exp_std = [], [], [], []  # Resultados promedio de cada experimento

    for _ in range(n_repetitions):
        puntos = np.random.uniform(size=(n_puntos, dim), low=-side, high=side)
        distances = pdist(puntos, metric='euclidean')

        exp_avg.append(np.mean(distances))
        exp_min.append(np.min(distances))
        exp_max.append(np.max(distances))
        exp_std.append(np.std(distances))

    # Promediar los experimentos
    avg_dist = np.mean(exp_avg)
    average_distances.append(avg_dist)
    min_distances.append(np.mean(exp_min))
    max_distances.append(np.mean(exp_max))
    std_distances.append(np.mean(exp_std))

    # Coeficiente de variación (desviación estándar relativa)
    relative_std.append(np.mean(exp_std) / avg_dist)


fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('La Maldición de la Dimensionalidad: Convergencia de Distancias', fontsize=16)

# 1. Distancias absolutas:
ax1.plot(dims, average_distances, 'b-', linewidth=2, label='Distancia promedio')
ax1.plot(dims, min_distances, 'g--', linewidth=2, label='Distancia mínima')
ax1.plot(dims, max_distances, 'r--', linewidth=2, label='Distancia máxima')
ax1.fill_between(dims, min_distances, max_distances, alpha=0.2, color='gray')
ax1.set_xlabel('Dimensiones')
ax1.set_ylabel('Distancia Euclidiana')
ax1.set_title('Evolución de las Distancias')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Razón máximo/mínimo:
ratio_max_min = (np.array(max_distances) - np.array(min_distances) )/ np.array(min_distances)
ax2.plot(dims, ratio_max_min, 'purple', linewidth=2)
ax2.set_xlabel('Dimensiones')
ax2.set_ylabel('Ratio Distancia Máx/Mín')
ax2.set_title('Convergencia: Ratio Distancia Máxima/Mínima')
ax2.grid(True, alpha=0.3)

# 3. Coeficiente de variación:
ax3.plot(dims, relative_std, 'orange', linewidth=2)
ax3.set_xlabel('Dimensiones')
ax3.set_ylabel('Coeficiente de Variación')
ax3.set_title('Variabilidad Relativa de las Distancias')
ax3.grid(True, alpha=0.3)

# 4. Distribución de distancias para dimensiones específicas
sample_dims = [3, 10, 100, 1000]
colors = ['blue', 'green', 'orange', 'red']

for dim, color in zip(sample_dims, colors):
    puntos_sample = np.random.uniform(size=(n_puntos, dim), low=-side, high=side)
    distances_sample = pdist(puntos_sample, metric='euclidean')
    ax4.hist(distances_sample,alpha=0.6, color=color,edgecolor='black',
            label=f'D={dim}', density=True, linewidth=0.5)

ax4.set_xlabel('Distancia Euclidiana')
ax4.set_ylabel('Densidad de Probabilidad')
ax4.set_title('Distribución de Distancias por Dimensionalidad')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



Esto nos dice que si tenemos *muchas* features, usar distancias entre vectores se vuelve algo cada vez menos significativo. **Reducir features** se vuelve muy útil para combatir este problema, podemos hacerlo:

1. **Seleccionando features** (selección de features): [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html), [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html), [`VarianceThreshold`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html)

2. **Proyectando en espacios de menor dimensión** (reducción de dimensionalidad): [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), [`TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html), [`UMAP`](https://umap-learn.readthedocs.io/), [`TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

En este modulo analizaremos estas estrategias para reducir la dimensionalidad.

# Ejercicios

1. Definir 3 vectores $u,v,w\in [-1,1]^D\subset \mathbb{R}^D$ con entradas aleatorias de una distribución uniforme en el rango $[-1,1]$. Calcula:

* Mide la distancia euclidiana entre ellos.
* Calcula
 $$\frac{d_{max}-d_{min}}{d_{min}}$$

 Usa $D=3,10,100,1000$.


2. Para cada uno de los valores de $D$ del ejercicio anterior, calcula el ángulo entre los vectores $u$ y $v$.

3. Repite el análisis de la dimensionalidad del inicio de la notebook usando las métricas:

 * Métrica Manthattan
 * Métrica angular

 Contesta lo siguiente: ¿Observas el mismo fenómeno que con la métrica Euclidiana?

4. Leer el conjunto $A$ de puntos del archivo `puntos_2d`.txt
 * Graficar los puntos
 * Encontrar una isometría $p:\mathbb{R}^2->\mathbb{R}$ para los puntos de $A$
 * Verificar que es una isometría
 * En caso de que el promedio de los puntos $p(A)\neq 0$ encontrar una isometría que sí satisfaga esta condición

In [ ]:
!wget https://raw.githubusercontent.com/gmauricio-toledo/tda-gdl/main/data/puntos_2d.txt

Observa el shape $N\times D$, donde $N$ es el número de puntos y $D$ es la dimensión del espacio  

In [ ]:
import numpy as np

puntos = np.loadtxt(fname='puntos_2d.txt')
print(f"Shape: {puntos.shape}")
puntos[:5,:]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs =  # Extraer las coordenadas x
ys =  # Extraer las coordenadas y

plt.figure()                    # Crea una nueva figura de gráfico
plt.scatter(xs, ys)             # Dibuja los puntos (x, y) como una nube de puntos
plt.axhline(y=0, color='black')     # Línea horizontal roja en y = 0 (eje X)
plt.axvline(x=0, color='black')     # Línea vertical roja en x = 0 (eje Y)
plt.axis('equal')               # Iguala la escala de los ejes x e y
plt.show()                      # Muestra el gráfico